In [2]:
import pandas as pd
# 特征转换 是对数据做数学 统计处理 使其适配模型输入要求
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
import numpy as np


In [3]:
# 归一化
x = np.array([[1, 2], [3, 4], [5, 6]])
scaler = MinMaxScaler() #   缩放到 【】
x_scaled = scaler.fit_transform(x)
print(x_scaled)



[[0.  0. ]
 [0.5 0.5]
 [1.  1. ]]


In [4]:
#  标准化  均值为0 方差为1
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)
print(x_scaled)


[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]


In [8]:
# 标签编码
from sklearn.preprocessing import LabelEncoder
# 要点： 一维、类数组（array-like）的"标签"数据，形状为 (n_samples,)。它的设计用途是编码目标标签 y，而不是特征矩阵 X。

encoder = LabelEncoder()
data = ["本科", "硕士", "博士", "本科"]  # 枚举值
result = encoder.fit_transform(data)
result

array([1, 2, 0, 1])

In [10]:
# 热编码
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder()

data = [[ "本科"], [ "硕士"], [ "博士"], ["本科"]] # 二维、类数组（array-like）的"特征表"，
result = encoder.fit_transform(data).toarray()
result


array([[0., 1., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       [0., 1., 0.]])

### 构造特征
>  基于现有的特征创造出新的特征  通过组合 、转换 、聚合 、现有特征


In [15]:
# 文本向量化
from sklearn.feature_extraction.text import CountVectorizer # 没啥用
vectorizer = CountVectorizer()

data = ["我爱北京天安门", "天安门上太阳升", "我爱北京天安门","安门","上海"]
result = vectorizer.fit_transform(data).toarray()
result


array([[0, 0, 0, 1],
       [0, 1, 0, 0],
       [0, 0, 0, 1],
       [0, 0, 1, 0],
       [1, 0, 0, 0]])

In [19]:
#  面积  总价 构造出单价 更有说服力
import pandas as pd
df = pd.DataFrame({
    "area": [50, 60, 70, 80, 90],
    "price": [50000, 60000, 70000, 80000, 90000]
})
df["unit_price"] = df["price"] / df["area"]
df


,area,price,unit_price
0,50,50000,1000.0
1,60,60000,1000.0
2,70,70000,1000.0
3,80,80000,1000.0
4,90,90000,1000.0


In [21]:
# 分箱  连续特征离散化
from sklearn.preprocessing import KBinsDiscretizer
# 不同年龄数据集
ages =  np.array([[18], [22], [25], [30], [35], [40], [45], [50], [55], [60], [65], [70], [75],[30], [35], [40], [45], [50], [55], [60], [65], [70], [75]])
est = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform')
print(est.fit_transform(ages))  # 不同年龄段的离散化结果




[[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [1.]
 [1.]
 [1.]
 [1.]
 [2.]
 [2.]
 [2.]
 [2.]
 [0.]
 [0.]
 [1.]
 [1.]
 [1.]
 [1.]
 [2.]
 [2.]
 [2.]
 [2.]]


In [22]:
# 4. 日期特征提取
dates = pd.DataFrame({"日期": pd.to_datetime(["2026-01-01", "2026-06-15", "2026-11-27"])})
dates["年"] = dates["日期"].dt.year
dates["月"] = dates["日期"].dt.month
dates["星期"] = dates["日期"].dt.dayofweek
dates["是否周末"] = dates["日期"].dt.dayofweek >= 5
print(dates)


          日期     年   月  星期   是否周末
0 2026-01-01  2026   1   3  False
1 2026-06-15  2026   6   0  False
2 2026-11-27  2026  11   4  False


/var/folders/d_/79_wtvyd1_dckhlrqzx0typ00000gn/T/ipykernel_79372/1959586433.py:5: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  dates["星期"] = dates["日期"].dt.dayofweek
/var/folders/d_/79_wtvyd1_dckhlrqzx0typ00000gn/T/ipykernel_79372/1959586433.py:6: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  dates["是否周末"] = dates["日期"].dt.dayofweek >= 5


# 特征降维
>  数据集特征多，保留主要特征，减少计算量

In [43]:
# 1. PCA 主成分分析（无监督，最常用
"""
思想：找数据方差最大的方向作为新坐标轴（方差大 = 信息多），依次取前 k 个方向，把数据投影上去
不看标签 y，纯看 X 的结构
"""
from pathlib import Path
base_path = Path.cwd()
print(base_path)
df = pd.read_csv(base_path / "data" / "heart_disease.csv")
df.head()

/Users/chunlin/Desktop/front-workspace/dl/04-sklearn


,年龄,性别,胸痛类型,静息血压,胆固醇,空腹血糖,静息心电图结果,最大心率,运动性心绞痛,运动后的ST下降,峰值ST段的斜率,主血管数量,地中海贫血,是否患有心脏病
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [47]:
#X=  df.drop(columns=df.columns[-1])
X=df.drop(columns=["是否患有心脏病"])
Y = df[df.columns[-1]]
print(X.head())

   年龄  性别  胸痛类型  静息血压  胆固醇  空腹血糖  静息心电图结果  最大心率  运动性心绞痛  运动后的ST下降  峰值ST段的斜率  \
0  52   1     0   125  212     0        1   168       0       1.0         2   
1  53   1     0   140  203     1        0   155       1       3.1         0   
2  70   1     0   145  174     0        1   125       1       2.6         0   
3  61   1     0   148  203     0        1   161       0       0.0         2   
4  62   0     0   138  294     1        1   106       0       1.9         1   

   主血管数量  地中海贫血  
0      2      3  
1      0      3  
2      0      3  
3      1      3  
4      3      2  


In [54]:
# pca 前一定要 数据归一化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(X_scaled.shape)


(1025, 13)


In [55]:
# 创建pca
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)
print(X_pca.shape)

(1025, 12)
